# Índice de Caminabilidad — Piloto Av. Roosevelt
**ITT Cali Inteligente · Equipo de Gobierno de Datos**

Este notebook calcula las métricas de caminabilidad para el área de influencia
de la intervención en Av. Roosevelt, usando la red peatonal de OpenStreetMap.

**Sistema de referencia:** WGS84 (EPSG:4326) para toda la información geoespacial.
Se usa EPSG:3116 (Colombia) únicamente para cálculos de área y distancia.

**Datos:** `data/itt_roosevelt/Roosevelt/Geojson_Roosevelt/`

**Instrucciones:**
1. Ejecutar las celdas en orden
2. Los datos se cargan automáticamente
3. Los resultados y mapas se generan automáticamente

## Celda 1 — Instalación de dependencias

In [6]:
import subprocess, sys
def check_pkg(pkg):
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for p in ['osmnx', 'geopandas', 'matplotlib', 'pandas', 'numpy', 'folium', 'seaborn']:
    check_pkg(p)
print('Dependencias verificadas')

Dependencias verificadas


## Celda 2 — Importaciones y configuración

In [7]:
import os, re, json, warnings, datetime, logging
from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd
import osmnx as ox
import matplotlib.pyplot as plt
import seaborn as sns
import folium
warnings.filterwarnings('ignore')
logging.getLogger('pyogrio').setLevel(logging.ERROR)

# Sistema de referencia
CRS_WGS84 = 'EPSG:4326'      # Sistema de trabajo (geográfico)
CRS_COLOMBIA = 'EPSG:3116'    # Solo para cálculos de área/distancia (proyectado)

plt.rcParams.update({
    'figure.facecolor': '#F4F6F9',
    'axes.facecolor': 'white',
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3
})
print(f'Sistema de referencia de trabajo: {CRS_WGS84}')
print(f'Sistema proyectado para cálculos: {CRS_COLOMBIA}')
print('Configuración lista')

Sistema de referencia de trabajo: EPSG:4326
Sistema proyectado para cálculos: EPSG:3116
Configuración lista


## Celda 3 — Parámetros, rutas y detección de entorno

In [11]:
# Repositorio del proyecto
REPO_URL = 'https://github.com/j0rg3c45/indice-caminabilidad-roosevelt.git'
REPO_NAME = 'indice-caminabilidad-roosevelt'

# Detectar entorno: Colab o local
if os.path.exists('/content'):
    import subprocess as _sp
    os.chdir('/content')
    _sp.getoutput(f'rm -rf {REPO_NAME}')
    print(_sp.getoutput(f'git clone {REPO_URL}'))
    PROJECT_ROOT = Path(f'/content/{REPO_NAME}')
else:
    PROJECT_ROOT = Path(os.getcwd()).parent

# Rutas relativas al proyecto
DATA_DIR = PROJECT_ROOT / 'data' / 'itt_roosevelt' / 'Roosevelt' / 'Geojson_Roosevelt'
IMG_DIR = PROJECT_ROOT / 'outputs' / 'figures'
RESULTS_DIR = PROJECT_ROOT / 'outputs' / 'results'

IMG_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Búsqueda flexible de archivos GeoJSON
def find_file(base_dir, pattern):
    regex = re.compile(pattern, re.IGNORECASE)
    for root, _dirs, files in os.walk(base_dir):
        for f in files:
            if regex.search(f) and f.lower().endswith('.geojson'):
                return os.path.join(root, f)
    return None

# Rutas a los archivos de datos
PATHS = {
    'poligono_buffer': find_file(DATA_DIR, r'Geojson_tramos_Roosevelt_Buffer_100'),
    'poligono_tramos': find_file(DATA_DIR, r'Geojson_tramos_Roosevelt\.geojson'),
    'siniestros':      find_file(DATA_DIR, r'BD_SINIESTROS'),
    'comparendos':     find_file(DATA_DIR, r'COMPARENDOS'),
    'homicidios':      find_file(DATA_DIR, r'HOMICIDIOS'),
    'hurtos':          find_file(DATA_DIR, r'HURTOS'),
    'sedes':           find_file(DATA_DIR, r'Sedes_educativas'),
    'vbg':             find_file(DATA_DIR, r'VBG'),
    'vif':             find_file(DATA_DIR, r'VIOLENCIA_INTRAFAMILIAR'),
}

ZONA_NOMBRE = 'Av. Roosevelt — Cali'

print(f'DATA_DIR: {DATA_DIR}')
print(f'IMG_DIR:  {IMG_DIR}')
print(f'RESULTS_DIR: {RESULTS_DIR}')
print()
print('Archivos encontrados:')
for nombre, ruta in PATHS.items():
    ok = ruta is not None and os.path.exists(ruta)
    estado = chr(9989) if ok else chr(10060)
    print(f'  {estado}  {nombre:18s}: {Path(ruta).name if ruta else "NO ENCONTRADO"}')

Cloning into 'indice-caminabilidad-roosevelt'...
DATA_DIR: /content/indice-caminabilidad-roosevelt/data/itt_roosevelt/Roosevelt/Geojson_Roosevelt
IMG_DIR:  /content/indice-caminabilidad-roosevelt/outputs/figures
RESULTS_DIR: /content/indice-caminabilidad-roosevelt/outputs/results

Archivos encontrados:
  ✅  poligono_buffer   : Geojson_tramos_Roosevelt_Buffer_100.geojson
  ✅  poligono_tramos   : Geojson_tramos_Roosevelt.geojson
  ✅  siniestros        : BD_SINIESTROS_2023_2025_COMUNA_BARRIO_4326_Roosevelt.geojson
  ✅  comparendos       : COMPARENDOS_2023_2025_Roosevelt.geojson
  ✅  homicidios        : HOMICIDIOS_2023_2025_Roosevelt.geojson
  ✅  hurtos            : HURTOS_2023_2025_Roosevelt.geojson
  ✅  sedes             : Sedes_educativas_oficiales_Roosevelt.geojson
  ✅  vbg               : VBG_2025_Roosevelt.geojson
  ✅  vif               : VIOLENCIA_INTRAFAMILIAR_2023_2025_Roosevelt.geojson


## Celda 4 — Cargar polígono y asegurar WGS84

In [12]:
# Cargar polígono de intervención (buffer 100m)
gdf_poligono = gpd.read_file(PATHS['poligono_buffer'])

# Asegurar que el polígono esté en WGS84
if gdf_poligono.crs is None:
    print('AVISO: GeoJSON sin CRS definido, asignando WGS84')
    gdf_poligono = gdf_poligono.set_crs(CRS_WGS84)
elif gdf_poligono.crs.to_epsg() != 4326:
    print(f'Reproyectando polígono de {gdf_poligono.crs} a WGS84...')
    gdf_poligono = gdf_poligono.to_crs(CRS_WGS84)

# Polígono en WGS84 para OSMnx y visualización
polygon_wgs84 = gdf_poligono.geometry.iloc[0]

# Proyectar a EPSG:3116 SOLO para cálculos de área/distancia
gdf_proyectado = gdf_poligono.to_crs(CRS_COLOMBIA)
area_m2 = gdf_proyectado.geometry.area.iloc[0]
area_km2 = area_m2 / 1_000_000

print(f'\n=== ÁREA DE INTERVENCIÓN ===')
print(f'  Zona: {ZONA_NOMBRE}')
print(f'  CRS de trabajo: {gdf_poligono.crs}')
print(f'  CRS para cálculos: {CRS_COLOMBIA}')
print(f'  Área: {area_m2:,.0f} m² ({area_m2/10000:.2f} ha)')
if 'BUFF_DIST' in gdf_poligono.columns:
    print(f'  Buffer aplicado: {gdf_poligono["BUFF_DIST"].iloc[0]:.0f} m a cada lado del eje')
if 'Shape_Leng' in gdf_poligono.columns:
    print(f'  Longitud aproximada del corredor: {gdf_poligono["Shape_Leng"].iloc[0]:.0f} m')


=== ÁREA DE INTERVENCIÓN ===
  Zona: Av. Roosevelt — Cali
  CRS de trabajo: EPSG:4326
  CRS para cálculos: EPSG:3116
  Área: 429,014 m² (42.90 ha)
  Buffer aplicado: 100 m a cada lado del eje
  Longitud aproximada del corredor: 1642 m


## Celda 5 — Descargar red peatonal desde OpenStreetMap

In [13]:
# OSMnx requiere el polígono en WGS84 (EPSG:4326)
# simplify=False conserva todos los vértices intermedios de las líneas
print(f'Descargando red peatonal (polígono en {CRS_WGS84})...')
G = ox.graph_from_polygon(polygon_wgs84, network_type='walk', simplify=False)
print(f'Red descargada: {len(G.nodes)} nodos, {len(G.edges)} segmentos')
print(f'CRS del grafo: EPSG:4326 (WGS84)')
print('Geometría: vértices intermedios conservados (simplify=False)')

Descargando red peatonal (polígono en EPSG:4326)...
Red descargada: 246 nodos, 762 segmentos


AttributeError: module 'osmnx.projection' has no attribute 'default_crs'

## Celda 6 — Calcular métricas de caminabilidad

In [ ]:
# Calcular estadísticas (área en m² para densidades correctas)
stats = ox.basic_stats(G, area=area_m2)

# Exportar red peatonal como GeoJSON en la carpeta de datos de la zona de estudio
gdf_nodos, gdf_aristas = ox.graph_to_gdfs(G)
red_peatonal_path = DATA_DIR / 'red_peatonal_osm_roosevelt.geojson'
gdf_aristas.to_crs(CRS_WGS84).to_file(red_peatonal_path, driver='GeoJSON')
print(f'Red peatonal guardada en zona de estudio: {red_peatonal_path}')

longitud_total_km = stats['edge_length_total'] / 1000
densidad_km_km2 = longitud_total_km / area_km2

metricas_caminabilidad = {
    'Intersecciones peatonales': stats['intersection_count'],
    'Longitud red peatonal (km)': round(longitud_total_km, 2),
    'Longitud promedio segmento (m)': round(stats['edge_length_avg'], 1),
    'Densidad calle (km/km²)': round(densidad_km_km2, 2),
    'Nodos OSM': len(G.nodes),
    'Segmentos OSM': len(G.edges),
}

print('=== MÉTRICAS DE CAMINABILIDAD — LÍNEA BASE ===')
print(f'  Fecha: {datetime.date.today().isoformat()}')
print(f'  CRS análisis: {CRS_WGS84} | Cálculos métricos: {CRS_COLOMBIA}')
print()
for k, v in metricas_caminabilidad.items():
    print(f'  {k}: {v}')

## Celda 7 — Cargar datos geoespaciales complementarios (WGS84)

In [ ]:
# Cargar todos los datasets y asegurar WGS84
datasets = {}
print('=== CARGA DE DATASETS (normalizados a WGS84) ===')
print()
for nombre, ruta in PATHS.items():
    if ruta and os.path.exists(ruta) and nombre not in ('poligono_buffer', 'poligono_tramos'):
        try:
            gdf = gpd.read_file(ruta)
            # Asegurar WGS84
            if gdf.crs is None:
                gdf = gdf.set_crs(CRS_WGS84)
            elif gdf.crs.to_epsg() != 4326:
                gdf = gdf.to_crs(CRS_WGS84)
            datasets[nombre] = gdf
            print(f'  {chr(9989)} {nombre:18s}: {len(gdf):>5} registros | CRS: {gdf.crs.to_epsg()}')
        except Exception as e:
            print(f'  {chr(10060)} {nombre:18s}: Error - {e}')

print(f'\nTotal datasets cargados: {len(datasets)}')
print(f'Todos en CRS: {CRS_WGS84}')

## Celda 8 — Mapa estático de la red peatonal

In [ ]:
fig, ax = ox.plot_graph(
    G,
    node_size=12,
    edge_linewidth=1.2,
    bgcolor='white',
    node_color='#2A9C8A',
    edge_color='#1A3A4A',
    figsize=(12, 12),
    show=False,
    close=False
)
ax.set_title(
    f'Red peatonal — {ZONA_NOMBRE}\nLínea base ITT · CRS: {CRS_WGS84}',
    fontsize=13, pad=15
)
plt.tight_layout()
plt.savefig(IMG_DIR / 'roosevelt_red_peatonal_linea_base.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Mapa guardado: {IMG_DIR / "roosevelt_red_peatonal_linea_base.png"}')

## Celda 9 — Mapa interactivo con capas (WGS84)

In [ ]:
# Mapa interactivo — todo en WGS84
centroid = polygon_wgs84.centroid
m = folium.Map(location=[centroid.y, centroid.x], zoom_start=15,
               tiles=None)

# Capas base
folium.TileLayer('CartoDB positron', name='CartoDB Claro').add_to(m)
folium.TileLayer('OpenStreetMap', name='OpenStreetMap').add_to(m)
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri', name='Esri Satélite', overlay=False
).add_to(m)
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}',
    attr='Esri', name='Esri Streets', overlay=False
).add_to(m)

# Polígono de intervención (ya en WGS84)
folium.GeoJson(
    gdf_poligono.__geo_interface__,
    name='Polígono intervención (Buffer 100m)',
    style_function=lambda x: {'color': '#1B4F8A', 'fillColor': '#2E7D32',
                              'fillOpacity': 0.1, 'weight': 2}
).add_to(m)

# Tramos de Roosevelt (eje del corredor)
if PATHS['poligono_tramos'] and os.path.exists(PATHS['poligono_tramos']):
    gdf_tramos = gpd.read_file(PATHS['poligono_tramos'])
    if gdf_tramos.crs is None:
        gdf_tramos = gdf_tramos.set_crs(CRS_WGS84)
    elif gdf_tramos.crs.to_epsg() != 4326:
        gdf_tramos = gdf_tramos.to_crs(CRS_WGS84)
    folium.GeoJson(
        gdf_tramos.__geo_interface__,
        name='Tramos Roosevelt (eje)',
        style_function=lambda x: {'color': '#E53935', 'weight': 4, 'opacity': 0.8}
    ).add_to(m)

# Red peatonal OSM (descargada en Celda 5)
gdf_edges = ox.graph_to_gdfs(G, nodes=False, edges=True)
gdf_edges_wgs84 = gdf_edges.to_crs(CRS_WGS84) if gdf_edges.crs.to_epsg() != 4326 else gdf_edges
folium.GeoJson(
    gdf_edges_wgs84[['geometry']].__geo_interface__,
    name='Red peatonal OSM',
    style_function=lambda x: {'color': '#2A9C8A', 'weight': 2, 'opacity': 0.7}
).add_to(m)

# Colores por capa
COLORES = {
    'siniestros': 'red', 'comparendos': 'orange', 'homicidios': 'darkred',
    'hurtos': 'purple', 'sedes': 'blue', 'vbg': 'pink', 'vif': 'darkpurple',
}

# Agregar capas (todas en WGS84)
for nombre, gdf in datasets.items():
    fg = folium.FeatureGroup(name=nombre.capitalize(), show=False)
    color = COLORES.get(nombre, 'gray')
    for _, row in gdf.iterrows():
        geom = row.geometry
        if geom is None:
            continue
        if geom.geom_type == 'Point':
            folium.CircleMarker(
                location=[geom.y, geom.x],
                radius=3, color=color, fill=True, fill_opacity=0.6
            ).add_to(fg)
    fg.add_to(m)

folium.LayerControl().add_to(m)
print(f'Mapa interactivo generado — CRS: {CRS_WGS84}')
m

## Celda 10 — Indicadores complementarios

In [ ]:
print('=== INDICADORES COMPLEMENTARIOS — LÍNEA BASE ===')
print(f'  Zona: {ZONA_NOMBRE}')
print(f'  Fecha: {datetime.date.today().isoformat()}')
print(f'  CRS: {CRS_WGS84} (cálculos de densidad con área en {CRS_COLOMBIA})')
print()

indicadores = {}
for nombre, gdf in datasets.items():
    n_registros = len(gdf)
    densidad = n_registros / (area_m2 / 10000)  # por hectárea
    indicadores[nombre] = {'total': n_registros, 'densidad_por_ha': round(densidad, 2)}
    print(f'  {nombre:18s}: {n_registros:>5} registros | {densidad:.2f} por ha')

print()
print('Nota: estos valores constituyen la línea base pre-intervención.')

## Celda 11 — Exportar resultados

In [ ]:
resultado = {
    'fecha_medicion': datetime.date.today().isoformat(),
    'poligono': 'Av. Roosevelt - Buffer 100m',
    'crs_trabajo': CRS_WGS84,
    'crs_calculo': CRS_COLOMBIA,
    'area_m2': round(area_m2, 0),
    'area_ha': round(area_m2 / 10000, 2),
    'intersecciones_peatonales': stats['intersection_count'],
    'longitud_red_peatonal_km': round(longitud_total_km, 2),
    'longitud_promedio_segmento_m': round(stats['edge_length_avg'], 1),
    'densidad_calle_km_km2': round(densidad_km_km2, 2),
    'nodos_osm': len(G.nodes),
    'segmentos_osm': len(G.edges),
    'fuente': 'OpenStreetMap via OSMnx',
    'momento': 'linea_base_pre_intervencion',
}

# Agregar conteos complementarios
for nombre, vals in indicadores.items():
    resultado[f'total_{nombre}'] = vals['total']
    resultado[f'densidad_{nombre}_por_ha'] = vals['densidad_por_ha']

df_resultados = pd.DataFrame([resultado])
output_csv = RESULTS_DIR / 'roosevelt_caminabilidad_linea_base.csv'
df_resultados.to_csv(output_csv, index=False)

print(f'Resultados exportados: {output_csv}')
print()
print(df_resultados.T.to_string())

## Celda 12 — Descarga (solo Google Colab)

In [ ]:
# Mostrar mapa como imagen en el notebook
from IPython.display import Image, display

mapa_path = IMG_DIR / 'roosevelt_red_peatonal_linea_base.png'

if mapa_path.exists():
    display(Image(filename=str(mapa_path), width=800))
else:
    print('Mapa no encontrado. Ejecuta primero la Celda 8.')

print(f'\nArchivos generados:')
print(f'  CSV: {output_csv}')
print(f'  Mapa: {mapa_path}')

# Descarga en Colab (comentado — descomentar si se necesita)
# if os.path.exists('/content'):
#     from google.colab import files
#     files.download(str(output_csv))
#     files.download(str(mapa_path))